In [ ]:
using LinearAlgebra
using PythonPlot

In [ ]:
#quaternion functions
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

function unhat(S)
    return 0.5*[S[3,2]-S[2,3];
                S[1,3]-S[3,1];
                S[2,1]-S[1,2]]
end

H = [zeros(1,3); I];
T = [1  zeros(1,3);
     zeros(3,1) -I];

function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I - hat(q[2:4])]
end

function G(q)
    return L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

function expq(ϕ)
    θ = norm(ϕ)
    return [cos(θ); ϕ*sinc(θ/π)];
end

In [ ]:
#dynamics
function dynamics(x)
    q = x[1:4]
    q = q/norm(q)
    
    ω = x[5:7]
    
    q̇ = 0.5*G(q)*ω
    ω̇ = -J\(hat(ω)*J*ω)

    ẋ = [q̇; ω̇]
end

In [ ]:
#Classic RK4 integrator: https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods
function rkstep(x)
    f1 = dynamics(x)
    f2 = dynamics(x + 0.5*h*f1)
    f3 = dynamics(x + 0.5*h*f2)
    f4 = dynamics(x + h*f3)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    xn[1:4] .= xn[1:4]/norm(xn[1:4]) #re-normalize quaternion
    return xn
end

In [ ]:
J = Diagonal([1.0; 1.25; 1.5])

In [ ]:
h = 0.1 #time step
n = 600 #number of time steps
tf = n*h #final time

#sample random initial attitude
q0 = randn(4)
q0 = q0/norm(q0)

#sample random angular velocity
ω0 = 0.1*randn(3)

#sample random gyro bias
β0 = randn(3)

x0 = [q0; ω0];

In [ ]:
#Simulate n time steps
xtraj = zeros(7,n)
xtraj[:,1] .= x0
for k = 1:(n-1)
    xtraj[:,k+1] = rkstep(xtraj[:,k])
end

In [ ]:
# Noise
m = 2 #number of vector observations
W = 0.01*I(3*m) #measurement noise
V = 0.01*I(6) #process noise

In [ ]:
#generate noisy gyro measurements
gyro = zeros(3,n)
for k = 1:n
    gyro[:,k] = xtraj[5:7] + β0 + sqrt(V[4:6,4:6])*randn(3)
end

In [ ]:
#generate random inertial vectors
r_N = zeros(3,m)
for ℓ = 1:m
    r_N[:,ℓ] = randn(3)
    r_N[:,ℓ] = r_N[:,ℓ]/norm(r_N[:,ℓ])
end

In [ ]:
#Generate noisy vector measurements
ytraj = zeros(3*m,n)
for k = 1:n
    Qk = Q(xtraj[1:4,k])
    yk = zeros(3,m)
    w = reshape(sqrt(W)*randn(3*m),3,m)
    for ℓ = 1:m
        Qw = exp(hat(w[:,ℓ]))
        yk[:,ℓ] = Qw'*Qk'*r_N[:,ℓ]
    end
    ytraj[:,k] .= yk[:]
end

In [ ]:
function state_prediction(x,u,h)
    q = x[1:4]
    β = x[5:7]
    
    return ##Fill me in
end

function state_prediction_deriv(x,u,h)
    q = x[1:4]
    β = x[5:7]


    return ##Fill me in
end

In [ ]:
function measurement_prediction(x,r_N)
    q = x[1:4]
    Qk = Q(q)
    y = Qk'*r_N
    return y[:]
end

function measurement_prediction_deriv(x,r_N)
    q = x[1:4]
    m = size(r_N,2)
    C = zeros(3*m,6)
    for k = 1:m
        C[((k-1)*3).+(1:3),1:3] .= H'*(L(q)'*L(H*r_N[:,k]) + R(q)*R(H*r_N[:,k])*T)*G(q)
    end
    return C
end

In [ ]:
# Filter Initialization
xfilt = zeros(7, n)
xfilt[1:4,1] = q0+0.1*randn(4)
xfilt[1:4,1] = xfilt[1:4,1]/norm(xfilt[1:4,1])

P = zeros(6,6,n)
P[:,:,1] .= 0.5*I(6);

In [ ]:
for k = 1:(n-1)
    #Prediction
    xpred = state_prediction(xfilt[:,k],gyro[:,k],h)
    A = state_prediction_deriv(xfilt[:,k],gyro[:,k],h)
    Ppred = A*P[:,:,k]*A' + V

    #Innovation
    z = ytraj[:,k+1] - measurement_prediction(xpred,r_N)
    C = measurement_prediction_deriv(xpred,r_N)
    S = C*Ppred*C' + W

    #Kalman Gain
    K = Ppred*C'/S

    #Update
    ##Fill me in
    xfilt[:,k+1] = ##Fill me in
    
    P[:,:,k+1] = (I(6) - K*C)*Ppred*(I(6) - K*C)' + K*W*K'
end

In [ ]:
plot(xfilt[1,:])
plot(xtraj[1,:])

In [ ]:
plot(xfilt[2,:])
plot(xtraj[2,:])

In [ ]:
plot(xfilt[3,:])
plot(xtraj[3,:])

In [ ]:
plot(xfilt[4,:])
plot(xtraj[4,:])

In [ ]:
plot(xfilt[5,:])
plot(β0[1]*ones(n))

In [ ]:
plot(xfilt[6,:])
plot(β0[2]*ones(n))

In [ ]:
plot(xfilt[7,:])
plot(β0[3]*ones(n))